# `process(StateChange)` — Yeatman2021 ROAR (full pipeline)

Production-pipeline counterpart to [`state_change.ipynb`](state_change.ipynb). Same induced-dyslexia experimental shape, but uses the actual `Yeatman2021-lexical_decision-image` benchmark and routes scoring through `bs_model.process()`'s full pipeline.

Four brain-score API calls:

```python
bs_model = brainscore.load_model('qwen2.5-vl-3b')
bs_model._state_change_fn = build_pytorch_ablation_fn(bs_model._model)
applied = bs_model.process(StateChange(...))
bs_model.reset()
```

Plus one call to score: `score = benchmark(bs_model)`.

**The lesion target.** We ablate **pseudo-selective units** (units more active for pseudoword images than real-word images at our chosen MLP layers). This is the empirically load-bearing direction in Qwen2.5-VL-3B — see §6 for the sweep that establishes this. It differs from Honarmand et al. (2026)'s framing, which targets word-vs-non-word-image-selective units; our localizer compares real vs pseudo *word* images so we identify a different class of units. Both produce real selective deficits.

**Hardware**: ~55 min on Mac MPS, ~15 min on EC2 g5.4xlarge.

In [ ]:
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

import brainscore
import brainscore_vision  # registers Qwen2.5-VL
from brainscore_core.model_interface import StateChange, Selection, Perturbation
from brainscore.perturbation import build_pytorch_ablation_fn


def resolve(model, path):
    """Walk a dotted path on a torch module."""
    for p in path.split('.'):
        model = model[int(p)] if p.isdigit() else getattr(model, p)
    return model


## 1 · Load the benchmark and model

In [ ]:
benchmark = brainscore.load_benchmark('Yeatman2021-lexical_decision-image')
bs_model = brainscore.load_model('qwen2.5-vl-3b')

# Wiring is explicit on purpose: brainscore_core stays torch-free, and
# non-pytorch models or custom ablation strategies pass their own
# callable here.
bs_model._state_change_fn = build_pytorch_ablation_fn(bs_model._model)

device = ('mps' if torch.backends.mps.is_available()
          else 'cuda' if torch.cuda.is_available() else 'cpu')
qwen = bs_model._model.to(device).eval()
processor = bs_model._preprocessors['vision']._processor

DYSLEXIA_THRESHOLD = 0.65  # paper's threshold (1 SD below human mean)

print(f'benchmark:    {benchmark.identifier}')
print(f'train stims:  {len(benchmark._train_stimuli)} (used as localizer)')
print(f'test stims:   {len(benchmark._test_stimuli)}')
print(f'human ceiling: {float(benchmark.ceiling):.3f}')
print(f'device:        {device}')


## 2 · Inspect the stimuli

In [ ]:
train_stim = benchmark._train_stimuli
real_paths   = train_stim[train_stim['image_label'] == 'real']['image_file_name'].tolist()
pseudo_paths = train_stim[train_stim['image_label'] == 'pseudo']['image_file_name'].tolist()

fig, axes = plt.subplots(2, 4, figsize=(11, 4))
for ax, p in zip(axes[0], real_paths[:4]):
    ax.imshow(Image.open(p)); ax.set_xticks([]); ax.set_yticks([])
for ax, p in zip(axes[1], pseudo_paths[:4]):
    ax.imshow(Image.open(p)); ax.set_xticks([]); ax.set_yticks([])
axes[0, 0].set_ylabel(f'real\n(n={len(real_paths)})',
                     rotation=0, ha='right', va='center', fontsize=11)
axes[1, 0].set_ylabel(f'pseudo\n(n={len(pseudo_paths)})',
                     rotation=0, ha='right', va='center', fontsize=11)
plt.suptitle('ROAR stimulus examples (4 train images per condition)', fontsize=11)
plt.tight_layout(); plt.show()


## 3 · Localize pseudo-selective units across late MLPs

We compute Cohen's d per unit at each of 5 late MLPs: `(μ_real − μ_pseudo) / σ_pooled`. **Bottom-K** of this distribution = units where mean(pseudo) > mean(real) = our 'pseudo-selective' targets. (Top-K = real-selective; we use those as a control later.)

Subsamples 50 real + 50 pseudo from the 200+200 train split — full set is ~25 min vs ~5 min subsampled and the contrast is captured at this scale.

In [ ]:
PROMPT_LOC = 'What word is shown in this image?'

def _generate_localize(image, max_new_tokens=1):
    msg = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image},
        {'type': 'text', 'text': PROMPT_LOC}]}]
    text = processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors='pt').to(device)
    with torch.no_grad():
        qwen.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

def localize_multi(real_imgs, pseudo_imgs, layer_paths):
    """Return per-layer Cohen's d selectivity vectors. Multi-hook so we
    capture all layers in one forward pass per stimulus."""
    captured = {lp: [] for lp in layer_paths}
    handles = []
    for lp in layer_paths:
        layer = resolve(qwen, lp)
        def hook(_m, _i, output, _lp=lp):
            h = output[0] if isinstance(output, tuple) else output
            captured[_lp].append(h.detach().to('cpu', dtype=torch.float32)
                                  .mean(dim=1).squeeze(0).numpy())
        handles.append(layer.register_forward_hook(hook))
    try:
        for img in real_imgs + pseudo_imgs:
            _generate_localize(img)
    finally:
        for h in handles: h.remove()
    selectivity_by_layer = {}
    for lp in layer_paths:
        acts = np.stack(captured[lp])
        real_act, pseudo_act = acts[:len(real_imgs)], acts[len(real_imgs):]
        pooled = np.sqrt((real_act.var(0) + pseudo_act.var(0)) / 2 + 1e-6)
        selectivity_by_layer[lp] = (real_act.mean(0) - pseudo_act.mean(0)) / pooled
    return selectivity_by_layer

random.seed(0)
LOC_N      = 50
loc_real   = [Image.open(p).convert('RGB') for p in random.sample(real_paths,   LOC_N)]
loc_pseudo = [Image.open(p).convert('RGB') for p in random.sample(pseudo_paths, LOC_N)]

LAYERS = [f'model.language_model.layers.{i}.mlp' for i in (26,28,30,32,34)]
TOP_K  = 500   # empirical sweet spot — see §6

selectivity_by_layer = localize_multi(loc_real, loc_pseudo, LAYERS)

# Bottom-K of selectivity = pseudo-selective units (our lesion target)
pseudo_selective_units = {L: np.argsort(selectivity_by_layer[L])[:TOP_K].tolist()
                          for L in LAYERS}

for L in LAYERS:
    s = selectivity_by_layer[L]
    print(f'  {L}  range [{s.min():+.2f}, {s.max():+.2f}]')


## 4 · Run the experiment via the full benchmark pipeline

`benchmark(bs_model)` is the production scoring path. Internally it does:
1. `bs_model.start_task(TaskContext(instruction=..., label_set=['real','pseudo']))`
2. `predictions = bs_model.process(test_stimuli)` — dispatches to `bs_model._generation_fn` (Qwen's chat-template + parse closure) for each test image
3. Computes accuracy, normalizes by ceiling, returns `Score`

The state_change hooks installed below are active during step 2's forward passes.

In [ ]:
# (a) baseline — no perturbation
baseline = benchmark(bs_model)
print(f'baseline:  raw={float(baseline.attrs["raw"]):.3f}  '
      f'real-acc={baseline.attrs["accuracy_real"]:.3f}  '
      f'pseudo-acc={baseline.attrs["accuracy_pseudo"]:.3f}')

# (b) install the lesion at every selected layer. Each process(StateChange)
# call dispatches to bs_model._state_change_fn, installs a forward hook,
# and registers a cleanup handle in bs_model._active_perturbations.
for layer, units in pseudo_selective_units.items():
    bs_model.process(StateChange(
        kind='ablation',
        target=Selection(layer=layer, indices=units),
        perturbation=Perturbation(kind='zero'),
    ))

# (c) score under lesion
lesioned = benchmark(bs_model)
print(f'lesioned:  raw={float(lesioned.attrs["raw"]):.3f}  '
      f'real-acc={lesioned.attrs["accuracy_real"]:.3f}  '
      f'pseudo-acc={lesioned.attrs["accuracy_pseudo"]:.3f}')

# (d) reset() invokes every cleanup in _active_perturbations and clears
# the registry. To remove a single perturbation without touching the
# others, use process(StateChange(kind='reset', handle_id=...)).
bs_model.reset()
restored = benchmark(bs_model)
print(f'restored:  raw={float(restored.attrs["raw"]):.3f}  '
      f'real-acc={restored.attrs["accuracy_real"]:.3f}  '
      f'pseudo-acc={restored.attrs["accuracy_pseudo"]:.3f}')

print()
print(f'lesioned dyslexic? {lesioned.attrs["dyslexic"]}  '
      f'(threshold = {DYSLEXIA_THRESHOLD})')


## 5 · Controls — does the selectivity actually matter?

Two controls demonstrate that the lesion above is hitting causally-relevant units rather than just generally damaging the layer:

- **Random control**: ablate the *same number* of randomly-chosen units at the same layers.
- **Real-selective control**: ablate the top-K *real-selective* units (the canonical / opposite-sign target).

If the random and real-selective controls produce scores close to baseline (~0.93) while the pseudo-selective lesion above breaks the model, that's evidence the lesion is selective. If they all break the model similarly, the lesion is just damage.

In [ ]:
def lesion_and_score(per_layer_units, label):
    """Install a multi-layer lesion, score via the benchmark, reset."""
    for layer, units in per_layer_units.items():
        bs_model.process(StateChange(
            kind='ablation',
            target=Selection(layer=layer, indices=units),
            perturbation=Perturbation(kind='zero'),
        ))
    s = benchmark(bs_model)
    bs_model.reset()
    print(f'{label}:  raw={float(s.attrs["raw"]):.3f}  '
          f'real-acc={s.attrs["accuracy_real"]:.3f}  '
          f'pseudo-acc={s.attrs["accuracy_pseudo"]:.3f}')
    return s

# Random control
random.seed(0)
n_units = selectivity_by_layer[LAYERS[0]].shape[0]
random_units = {L: random.sample(range(n_units), TOP_K) for L in LAYERS}
random_score = lesion_and_score(random_units, 'random         ')

# Real-selective control — top-K of selectivity = the OPPOSITE direction
real_selective_units = {L: np.argsort(selectivity_by_layer[L])[-TOP_K:].tolist()
                        for L in LAYERS}
real_score = lesion_and_score(real_selective_units, 'real-selective ')


## 6 · Why K=500 + pseudo-selective: the regime sweep

The defaults above (TOP_K=500, lesion the bottom-K of Cohen's d) come from a `K`-sweep that I ran offline. The full results:

| K | canonical (real-sel) | random | **pseudo-sel** | gap (random − can.) | gap (pseudo-sel − can.) |
|---|---|---|---|---|---|
| **500** | **0.92** | **0.91** | **0.50** | **−0.01** | **−0.42** |
| 750 | 0.92 | 0.52 | 0.50 | −0.40 | −0.42 |
| 1000 | 0.86 | 0.75 | 0.51 | −0.11 | −0.35 |
| 1250 | 0.58 | 0.61 | 0.50 | +0.03 | −0.08 |

**At K=500 selectivity is doing real work**: random and real-selective ablations both stay near baseline (0.91, 0.92), while pseudo-selective ablation breaks the model (0.50). The 0.42 score gap between pseudo-selective and the controls is the causal contribution of those specific units.

**At K=1250 selectivity stops mattering**: all three conditions converge to ~0.5–0.6. The lesion is so large that any choice of 1250 units (∼60% of a 2048-unit MLP) produces similar damage. This is the regime where top-K and bottom-K of Cohen's d overlap by ∼22%, so the selection sign loses leverage.

K=500 (∼25% of layer) keeps top and bottom of selectivity fully disjoint, so the sign is doing causal work. Sweep code: `unified/scripts/sweep_topk.py`.

## 7 · Visualize

In [ ]:
fig = plt.figure(figsize=(13, 4.5))
gs = fig.add_gridspec(1, 2, width_ratios=[2, 3], wspace=0.3)
ax_a = fig.add_subplot(gs[0]); ax_b = fig.add_subplot(gs[1])

# Panel A — Cohen's d selectivity at the last lesioned MLP, with the two
# tails (top-K real-selective and bottom-K pseudo-selective) shaded.
last = LAYERS[-1]
sel  = selectivity_by_layer[last]
ax_a.hist(sel, bins=80, color='#cccccc', edgecolor='white', linewidth=0.4)
for ix, color, label in [
        (np.argsort(sel)[-TOP_K:], '#3aa55a', f'real-selective top {TOP_K}'),
        (np.argsort(sel)[:TOP_K],  '#a54a4a', f'pseudo-selective bottom {TOP_K}')]:
    ax_a.hist(sel[ix], bins=40, color=color, alpha=0.7, label=label)
ax_a.set_xlabel(r'$\bar{a}_\mathrm{real} - \bar{a}_\mathrm{pseudo}$ (Cohen$\,d$)')
ax_a.set_ylabel('Units')
ax_a.set_title('A. Selectivity at last lesioned MLP', loc='left', fontsize=11)
ax_a.legend(fontsize=9)

# Panel B — accuracy bars across conditions
conds  = ['baseline',
          'pseudo-selective\n(main lesion)',
          'random control',
          'real-selective\n(opposite-sign control)',
          'restored']
scores = [baseline, lesioned, random_score, real_score, restored]
raw    = [float(s.attrs['raw'])         for s in scores]
real_a = [s.attrs['accuracy_real']      for s in scores]
pseudo_a = [s.attrs['accuracy_pseudo']  for s in scores]

x = np.arange(len(conds)); w = 0.27
ax_b.bar(x - w, raw,      width=w, color='#4a6fa5', label='overall')
ax_b.bar(x,     real_a,   width=w, color='#3aa55a', label='real-word')
ax_b.bar(x + w, pseudo_a, width=w, color='#a54a4a', label='pseudo-word')
ax_b.axhline(DYSLEXIA_THRESHOLD, color='black', lw=0.6, ls=':')
ax_b.text(len(conds) - 0.5, DYSLEXIA_THRESHOLD + 0.02,
          f'dyslexia threshold ({DYSLEXIA_THRESHOLD})', ha='right', fontsize=9)
ax_b.set_xticks(x); ax_b.set_xticklabels(conds, fontsize=9)
ax_b.set_ylabel('Accuracy'); ax_b.set_ylim(0, 1.08)
ax_b.set_title('B. Yeatman2021 lexical decision  '
               f'(n_test={float(baseline.attrs["n_test_stimuli"]):.0f})',
               loc='left', fontsize=11)
ax_b.legend(loc='lower right', fontsize=9)
for i, r in enumerate(raw):
    ax_b.text(i - w, r + 0.02, f'{r:.2f}', ha='center', fontsize=9)
for ax in (ax_a, ax_b):
    for sp in ('top', 'right'): ax.spines[sp].set_visible(False)
plt.show()


## What this tells us

**Three findings the controls let us make:**

1. **Real-word recognition is robust to ablation.** Across every condition (including 500-unit pseudo-selective lesion), `real-acc = 1.00`. The model never loses the ability to identify real words — Qwen's lexical knowledge is broadly distributed.

2. **Pseudo-word detection is causally concentrated.** Ablating just 500 specific units per layer (out of 2048) drops pseudo-acc from 0.86 → 0.00. Random-unit ablation at the same magnitude doesn't do it (0.82); real-selective ablation doesn't do it (0.86). The lesion is hitting a specific, identifiable circuit.

3. **The lesion crosses the 0.65 dyslexia threshold (0.50 < 0.65)**, satisfying the paper's quantitative criterion for an induced reading deficit.

## Comparison to the paper

Honarmand et al. (2026, ICLR) describe induced dyslexia as `real-acc drops, pseudo-acc preserved` — the canonical 'cannot recognize real words' pattern. We see the *opposite* default: pseudo-acc collapses, real-acc preserved. Both are valid signatures of impaired lexical decision; they differ in which direction the failure defaults.

**Why the difference?** Localizer framing.

- **Paper's localizer**: contrasts *word images* (real + pseudo together) vs *non-word images* (line drawings, scrambled stimuli, line faces). Identifies units that respond to text-of-any-kind. Ablating those impairs the model's ability to recognize printed text at all → real-words misclassified as pseudo (the canonical dyslexia direction).
- **Our localizer**: contrasts real-word images vs pseudo-word images (both are text). Identifies units that *discriminate* real from pseudo. The pseudo-selective subset turns out to be the load-bearing direction in Qwen2.5-VL-3B — ablating those removes the model's only path to the 'pseudo' answer → defaults to 'real'.

The paper's localizer requires non-text control stimuli (line drawings, etc.); our setup uses ROAR's existing real/pseudo split. The same brain-score `process(StateChange)` machinery accommodates either localizer — just plug different stimuli into `localize_multi`.

## What the framework demonstrated

Independent of *which* lesion direction you target, this notebook validates:
- `bs_model.process(StateChange(...))` × 5 calls accumulate handles in `_active_perturbations`.
- `bs_model.reset()` cleans them all up bit-for-bit (restored = baseline).
- Targeted lesions show the brain-score API surfaces controllable causal interventions through `process(input_event)` dispatch.

Same five lines of API code work for the toy demo, the full ROAR benchmark, and any future induced-dysfunction experiment that wants to fit the same pattern.